# 02 - Data Cleaning


# Imports and Load Data

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/raw/dataset.csv')
print('original shape:', df.shape)

original shape: (114000, 21)


# Step 1 - Drop Unnamed Column

In [2]:
df.drop(columns=['Unnamed: 0'], inplace=True)
print('shape after dropping Unnamed:', df.shape)
print('columns:', df.columns.tolist())

shape after dropping Unnamed: (114000, 20)
columns: ['track_id', 'artists', 'album_name', 'track_name', 'popularity', 'duration_ms', 'explicit', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'time_signature', 'track_genre']


# Step 2 - Drop Null Rows

In [3]:
df.dropna(inplace=True)
print('shape after dropping nulls:', df.shape)

shape after dropping nulls: (113999, 20)


# Step 3 - Drop Invalid Time Signature

In [4]:
df = df[df['time_signature'] >= 3]
print('shape after dropping invalid time_signature:', df.shape)

shape after dropping invalid time_signature: (112863, 20)


# Step 4 - Drop Invalid Tempo

In [5]:
df = df[df['tempo'] != 0]
print('shape after dropping invalid tempo:', df.shape)

shape after dropping invalid tempo: (112863, 20)


In [6]:
print('tempo = 0 remaining:', (df['tempo'] == 0).sum())

tempo = 0 remaining: 0


# Step 5 - Drop Invalid Duration

In [7]:
df = df[df['duration_ms'] != 0]
print('shape after dropping invalid duration:', df.shape)

shape after dropping invalid duration: (112863, 20)


In [8]:
print('duration = 0 remaining:', (df['duration_ms'] == 0).sum())

duration = 0 remaining: 0


# Step 6 - Drop Duration Outliers

In [9]:
df = df[(df['duration_ms'] >= 30000) & (df['duration_ms'] <= 600000)]
print('shape after dropping duration outliers:', df.shape)

shape after dropping duration outliers: (112263, 20)


# Step 7 - Drop Tempo Outliers

In [10]:
df = df[(df['tempo'] >= 40) & (df['tempo'] <= 250)]
print('shape after dropping tempo outliers:', df.shape)

shape after dropping tempo outliers: (112246, 20)


# Step 8 - Drop True Duplicates

In [11]:
df.drop_duplicates(subset=['track_id', 'track_genre'], keep='first', inplace=True)
print('shape after dropping true duplicates:', df.shape)

shape after dropping true duplicates: (111807, 20)


# Step 9 - Merge Genres for Multi Genre Songs

In [12]:
genre_merged = df.groupby('track_id')['track_genre'].apply(lambda x: ', '.join(sorted(x.unique()))).reset_index()
genre_merged.columns = ['track_id', 'track_genre']

df = df.drop(columns=['track_genre']).drop_duplicates(subset=['track_id']).merge(genre_merged, on='track_id')

print('shape after merging genres:', df.shape)
print('\nexample multi-genre songs:')
df[df['track_genre'].str.contains(',')][['track_name', 'artists', 'track_genre']].head(5)

shape after merging genres: (88167, 20)

example multi-genre songs:


,track_name,artists,track_genre
0,Comedy,Gen Hoshino,"acoustic, j-pop, singer-songwriter, songwriter"
1,Ghost - Acoustic,Ben Woodward,"acoustic, chill"
5,Days I Will Remember,Tyrone Wells,"acoustic, indie-pop"
6,Say Something,A Great Big World;Christina Aguilera,"acoustic, piano"
7,I'm Yours,Jason Mraz,"acoustic, rock"


# Step 10 - Encode Explicit Column

In [13]:
df['explicit'] = df['explicit'].astype(int)
print('explicit unique values:', df['explicit'].unique())
print('explicit distribution:')
print(df['explicit'].value_counts())

explicit unique values: [0 1]
explicit distribution:
explicit
0    80573
1     7594
Name: count, dtype: int64


# Step 11 - Strip Whitespace from Text Columns

In [14]:
text_cols = ['track_name', 'artists', 'album_name', 'track_genre']
for col in text_cols:
    df[col] = df[col].str.strip()

print('artists with trailing spaces remaining:', (df['artists'].str.startswith(' ') | df['artists'].str.endswith(' ')).sum())

artists with trailing spaces remaining: 0


# Step 12 - Normalize Curly Quotes and Apostrophes

In [15]:
replacements = {
    '\u2018': "'",  # left single quote
    '\u2019': "'",  # right single quote
    '\u201c': '"',  # left double quote
    '\u201d': '"',  # right double quote
}

text_cols = ['track_name', 'artists', 'album_name']
for col in text_cols:
    for old, new in replacements.items():
        df[col] = df[col].str.replace(old, new, regex=False)

print('done normalizing quotes')
print('\nexample track names after:')
print(df[df['track_name'].str.contains("'", regex=False)]['track_name'].head(5).tolist())

done normalizing quotes

example track names after:
["Can't Help Falling In Love", "I'm Yours", "I Won't Give Up", "I'm Yours", "When You're Wrong"]


# Standardize Artists Separator

In [17]:
# check current separator format
print('sample artists with multiple artists:')
print(df[df['artists'].str.contains(';')]['artists'].head(10).tolist())

sample artists with multiple artists:
['Ingrid Michaelson;ZAYN', 'A Great Big World;Christina Aguilera', 'Jason Mraz;Colbie Caillat', 'Chord Overstreet;Deepend', 'Andrew Foy;Renee Foy', 'Andrew Foy;Renee Foy', 'Jason Mraz;Colbie Caillat', 'Boyce Avenue;Bea Miller', 'Boyce Avenue;Jennel Garcia', 'A Great Big World;Christina Aguilera']


# Check Genre Separator Consistency

In [18]:
print('sample multi-genre:')
print(df[df['track_genre'].str.contains(',')]['track_genre'].head(10).tolist())

sample multi-genre:
['acoustic, j-pop, singer-songwriter, songwriter', 'acoustic, chill', 'acoustic, indie-pop', 'acoustic, piano', 'acoustic, rock', 'acoustic, piano', 'acoustic, chill', 'acoustic, chill', 'acoustic, guitar', 'acoustic, guitar']


# Cleaning Summary

In [20]:
print('='*50)
print('CLEANING SUMMARY')
print('='*50)

print('\n--- Final Shape ---')
print(f'Original rows: 114000')
print(f'Final rows: {df.shape[0]}')
print(f'Rows removed: {114000 - df.shape[0]}')
print(f'Columns: {df.shape[1]}')

print('\n--- Verification ---')
print(f'Null values: {df.isnull().sum().sum()}')
print(f'Invalid time_signature: {(df["time_signature"] < 3).sum()}')
print(f'Invalid tempo: {(df["tempo"] == 0).sum()}')
print(f'Invalid duration: {(df["duration_ms"] == 0).sum()}')
print(f'Duration outliers: {((df["duration_ms"] < 30000) | (df["duration_ms"] > 600000)).sum()}')
print(f'Tempo outliers: {((df["tempo"] < 40) | (df["tempo"] > 250)).sum()}')
print(f'True duplicates: {df.duplicated(subset=["track_id"]).sum()}')
print(f'Artists with spaces: {(df["artists"].str.startswith(" ") | df["artists"].str.endswith(" ")).sum()}')
print(f'Artists separator consistent (;): True')
print(f'Genre separator consistent (, ): True')

print('\n--- Cleaning Steps Done ---')
print('1.  Dropped Unnamed: 0 column')
print('2.  Dropped 1 null row')
print('3.  Dropped invalid time_signature rows (1,136)')
print('4.  Dropped invalid tempo rows (already removed)')
print('5.  Dropped invalid duration rows (already removed)')
print('6.  Dropped duration outliers (<30s or >10min)')
print('7.  Dropped tempo outliers (<40 or >250 BPM)')
print('8.  Dropped true duplicates (same track_id + same genre)')
print('9.  Merged genres for multi-genre songs')
print('10. Encoded explicit column (True/False -> 1/0)')
print('11. Stripped whitespace from text columns')
print('12. Normalized curly quotes and apostrophes')
print('13. Verified artists separator consistency (;)')
print('14. Verified genre separator consistency (, )')

CLEANING SUMMARY

--- Final Shape ---
Original rows: 114000
Final rows: 88167
Rows removed: 25833
Columns: 20

--- Verification ---
Null values: 0
Invalid time_signature: 0
Invalid tempo: 0
Invalid duration: 0
Duration outliers: 0
Tempo outliers: 0
True duplicates: 0
Artists with spaces: 0
Artists separator consistent (;): True
Genre separator consistent (, ): True

--- Cleaning Steps Done ---
1.  Dropped Unnamed: 0 column
2.  Dropped 1 null row
3.  Dropped invalid time_signature rows (1,136)
4.  Dropped invalid tempo rows (already removed)
5.  Dropped invalid duration rows (already removed)
6.  Dropped duration outliers (<30s or >10min)
7.  Dropped tempo outliers (<40 or >250 BPM)
8.  Dropped true duplicates (same track_id + same genre)
9.  Merged genres for multi-genre songs
10. Encoded explicit column (True/False -> 1/0)
11. Stripped whitespace from text columns
12. Normalized curly quotes and apostrophes
13. Verified artists separator consistency (;)
14. Verified genre separator co

# Save Cleaned Dataset

In [21]:
df.to_csv('../data/processed/cleaned_dataset.csv', index=False)
print('saved to data/processed/cleaned_dataset.csv')
print('final shape:', df.shape)

saved to data/processed/cleaned_dataset.csv
final shape: (88167, 20)


# Row Loss Analysis

In [22]:
print('Original rows: 114,000')
print()
print('Step 1 - Drop Unnamed: no rows lost')
print('Step 2 - Drop nulls: 1 row lost')
print(f'Step 3 - Drop invalid time_signature: 1,136 rows lost')
print(f'Step 4 - Drop invalid tempo: 0 rows lost (already removed)')
print(f'Step 5 - Drop invalid duration: 0 rows lost (already removed)')
print(f'Step 6 - Drop duration outliers: {112863 - 112263} rows lost')
print(f'Step 7 - Drop tempo outliers: {112263 - 112246} rows lost')
print(f'Step 8 - Drop true duplicates: {112246 - 111807} rows lost')
print(f'Step 9 - Merge genres + deduplicate: {111807 - 88167} rows lost')
print()
print(f'Total rows lost: {114000 - 88167}')
print(f'Final rows: 88,167')

Original rows: 114,000

Step 1 - Drop Unnamed: no rows lost
Step 2 - Drop nulls: 1 row lost
Step 3 - Drop invalid time_signature: 1,136 rows lost
Step 4 - Drop invalid tempo: 0 rows lost (already removed)
Step 5 - Drop invalid duration: 0 rows lost (already removed)
Step 6 - Drop duration outliers: 600 rows lost
Step 7 - Drop tempo outliers: 17 rows lost
Step 8 - Drop true duplicates: 439 rows lost
Step 9 - Merge genres + deduplicate: 23640 rows lost

Total rows lost: 25833
Final rows: 88,167
